In [24]:
import asyncio
import aiohttp
import re
import shutil
import time
from pathlib import Path

from google.colab import files

# ---------- CONFIG ----------
BASE_URL = "https://piece-climatisation.com"
START_URL = BASE_URL + "/boutique-piece-detachee-climatisation/fournisseur/lg"

OUTPUT_DIR = Path("lg_pages")
OUTPUT_DIR.mkdir(exist_ok=True)

CONCURRENT_REQUESTS = 15
RETRIES = 3
START_PAGE = 1   # 👈 resume from here

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120 Safari/537.36"
}


# ---------- HELPERS ----------
def safe_filename(page_num):
    return f"lg_page_{page_num}.html"


def extract_last_page(html):
    matches = re.findall(r'page=(\d+)', html)
    return max(map(int, matches)) if matches else 1


# ---------- FETCH ----------
async def fetch(session, url):
    for attempt in range(RETRIES):
        try:
            async with session.get(url, headers=HEADERS, timeout=30) as res:
                if res.status == 200:
                    return await res.text()
                else:
                    print(f"Status {res.status}: {url}")
        except Exception as e:
            print(f"Retry {attempt+1} → {url} → {e}")

        await asyncio.sleep(1)

    return None


# ---------- SAVE ----------
def save_html(page_num, html):
    file_path = OUTPUT_DIR / safe_filename(page_num)

    # ⏭ Skip if already exists (resume-safe)
    if file_path.exists():
        print(f"⏭ Skipped (exists): page {page_num}")
        return

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(html)

    print(f"✅ Saved page {page_num}")


# ---------- WORKER ----------
async def scrape_page(session, page_num, semaphore):
    async with semaphore:
        file_path = OUTPUT_DIR / safe_filename(page_num)

        # ⏭ Skip before request (faster resume)
        if file_path.exists():
            print(f"⏭ Already downloaded: {page_num}")
            return

        url = f"{START_URL}?page={page_num}" if page_num > 1 else START_URL

        html = await fetch(session, url)
        if html:
            save_html(page_num, html)
        else:
            print(f"❌ Failed page {page_num}")


# ---------- MAIN ----------
async def main():
    connector = aiohttp.TCPConnector(limit=CONCURRENT_REQUESTS)
    semaphore = asyncio.Semaphore(CONCURRENT_REQUESTS)

    async with aiohttp.ClientSession(connector=connector) as session:
        print("Fetching first page to detect total pages...")

        first_html = await fetch(session, START_URL)
        if not first_html:
            print("❌ Failed first page")
            return

        last_page = extract_last_page(first_html)
        print(f"Total pages: {last_page}")
        print(f"Resuming from page: {START_PAGE}")

        # Ensure start is valid
        start = max(1, START_PAGE)

        # Create tasks from resume point
        tasks = [
            scrape_page(session, i, semaphore)
            for i in range(start, last_page + 1)
        ]

        # Batch processing (important for 5000+ pages)
        batch_size = 150

        for i in range(0, len(tasks), batch_size):
            batch = tasks[i:i + batch_size]
            print(f"\n🚀 Batch {i + start} → {i + start + len(batch) - 1}")

            await asyncio.gather(*batch)

            # small delay to avoid blocking
            await asyncio.sleep(2)

    zip_file_name = 'lg_pages.zip'
    shutil.make_archive(zip_file_name.replace('.zip', ''), 'zip', 'lg_pages')

    print(f'Successfully created {zip_file_name}. Downloading now...')
    files.download(zip_file_name)


if __name__ == "__main__":
    await main()

Fetching first page to detect total pages...
Total pages: 3
Resuming from page: 1

🚀 Batch 1 → 3
✅ Saved page 1
✅ Saved page 3
✅ Saved page 2
Successfully created lg_pages.zip. Downloading now...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [25]:
import os
import json
import csv
from bs4 import BeautifulSoup

INPUT_FOLDER = "lg_pages"
BASE_URL = "https://piece-climatisation.com"  # <-- change this

OUTPUT_JSON = "lg_products.json"
OUTPUT_CSV = "lg_products.csv"


def clean_price(price_text):
    if not price_text:
        return ""
    return (
        price_text.replace("\xa0", " ")
        .replace("€", "")
        .replace("TTC", "")
        .strip()
    )


def parse_html_file(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "html.parser")

    products = []

    items = soup.select(".product-list-element")

    for item in items:
        try:
            # product name
            name_tag = item.select_one(".product-name b")
            name = name_tag.get_text(strip=True) if name_tag else ""

            # product URL
            link_tag = item.select_one(".product-name")
            relative_url = link_tag["href"] if link_tag else ""
            product_url = BASE_URL + relative_url if relative_url else ""

            # reference + brand
            ref_brand_text = item.select_one(".description p")
            ref = ""
            brand = ""

            if ref_brand_text:
                lines = ref_brand_text.get_text("\n").split("\n")
                for line in lines:
                    line = line.strip()
                    if line.startswith("Ref:"):
                        ref = line.replace("Ref:", "").strip()
                    elif line:
                        brand = line.strip()

            # price
            price_tag = item.select_one(".final-price")
            price = clean_price(price_tag.get_text()) if price_tag else ""

            products.append({
                "product_name": name,
                "product_url": product_url,
                "brand": brand,
                "reference": ref,
                "price": price
            })

        except Exception as e:
            print(f"Error parsing item in {filepath}: {e}")

    return products


def main():
    all_products = []

    files = [f for f in os.listdir(INPUT_FOLDER) if f.endswith(".html")]
    files.sort()  # ensure consistent order

    print(f"📂 Total files: {len(files)}")

    for idx, filename in enumerate(files, 1):
        filepath = os.path.join(INPUT_FOLDER, filename)
        print(f"Processing {idx}/{len(files)}: {filename}")

        products = parse_html_file(filepath)
        all_products.extend(products)

    print(f"\n✅ Total products scraped: {len(all_products)}")

    # Save JSON
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(all_products, f, indent=2, ensure_ascii=False)

    # Save CSV
    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=[
            "product_name",
            "product_url",
            "brand",
            "reference",
            "price"
        ])
        writer.writeheader()
        writer.writerows(all_products)

    print(f"\n💾 Saved to {OUTPUT_JSON} and {OUTPUT_CSV}")


if __name__ == "__main__":
    main()

📂 Total files: 3
Processing 1/3: lg_page_1.html
Processing 2/3: lg_page_2.html
Processing 3/3: lg_page_3.html

✅ Total products scraped: 25

💾 Saved to lg_products.json and lg_products.csv


In [19]:
import zipfile

zip_file_path = '/content/panasonic1_pages.zip'
extract_dir = '/content/panasonic1_pages'

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"Successfully unzipped {zip_file_path} to {extract_dir}")

Successfully unzipped /content/panasonic1_pages.zip to /content/panasonic1_pages


In [ ]:
"""
scraper_colab.py  —  Multi-brand HTML page scraper
====================================================
• Scrapes every page for each brand from piece-climatisation.com
• Saves HTML files into per-brand folders
• Zips + downloads each brand folder when it finishes
• Zips + downloads ALL brand folders together at the very end

Designed to run in Google Colab (uses google.colab.files for downloads).

Usage (in a Colab cell):
    exec(open("scraper_colab.py").read())
    await run_all()
"""

import asyncio
import aiohttp
import re
import shutil
import time
from pathlib import Path

from google.colab import files  # Colab-specific download

# ============================================================
# CONFIGURATION
# ============================================================

BASE_URL = "https://piece-climatisation.com"

# Full list of brands — add / remove as needed
BRANDS = [
    {"brand": "CARRIER TRANSICOLD",        "url": "/boutique-piece-detachee-climatisation/fournisseur/carrier-transicold"},
    {"brand": "AIRCALO",                   "url": "/boutique-piece-detachee-climatisation/fournisseur/aircalo"},
    {"brand": "FRANCE AIR",                "url": "/boutique-piece-detachee-climatisation/fournisseur/france-air"},
    {"brand": "THERMOR",                   "url": "/boutique-piece-detachee-climatisation/fournisseur/thermor"},
    {"brand": "SAMSUNG",                   "url": "/boutique-piece-detachee-climatisation/fournisseur/samsung"},
    {"brand": "HAIER",                     "url": "/boutique-piece-detachee-climatisation/fournisseur/haier"},
    {"brand": "YACK",                      "url": "/boutique-piece-detachee-climatisation/fournisseur/yack"},
    {"brand": "VIESSMAN",                  "url": "/boutique-piece-detachee-climatisation/fournisseur/viessman"},
    {"brand": "LTB",                       "url": "/boutique-piece-detachee-climatisation/fournisseur/ltb"},
    {"brand": "SICCOM",                    "url": "/boutique-piece-detachee-climatisation/fournisseur/siccom"},
    {"brand": "DE DIETRICH",               "url": "/boutique-piece-detachee-climatisation/fournisseur/de-dietrich"},
    {"brand": "OVENTROP",                  "url": "/boutique-piece-detachee-climatisation/fournisseur/oventrop"},
    {"brand": "SWEGON",                    "url": "/boutique-piece-detachee-climatisation/fournisseur/swegon"},
    {"brand": "CHAPPEE",                   "url": "/boutique-piece-detachee-climatisation/fournisseur/chappee"},
    {"brand": "CARRIER",                   "url": "/boutique-piece-detachee-climatisation/fournisseur/carrier"},
    {"brand": "DAIKIN",                    "url": "/boutique-piece-detachee-climatisation/fournisseur/daikin"},
    {"brand": "RIELLO",                    "url": "/boutique-piece-detachee-climatisation/fournisseur/riello"},
    {"brand": "ATLANTIC FUJITSU GENERAL",  "url": "/boutique-piece-detachee-climatisation/fournisseur/atlantic"},
    {"brand": "TRANE",                     "url": "/boutique-piece-detachee-climatisation/fournisseur/trane"},
    {"brand": "TECHNIBEL NIBE",            "url": "/boutique-piece-detachee-climatisation/fournisseur/technibel"},
    {"brand": "LENNOX",                    "url": "/boutique-piece-detachee-climatisation/fournisseur/lennox"},
    {"brand": "LG",                        "url": "/boutique-piece-detachee-climatisation/fournisseur/lg"},
  ]

# Scraper settings
CONCURRENT_REQUESTS = 15    # parallel requests per brand
BATCH_SIZE          = 100  # pages per asyncio.gather batch
RETRIES             = 3    # per-request retry count
RETRY_DELAY         = 1.5  # seconds between retries
BATCH_DELAY         = 2.0  # seconds between batches

# Root folder that holds all brand subfolders
ROOT_DIR  = Path("all_brands")
ROOT_DIR.mkdir(exist_ok=True)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "fr-FR,fr;q=0.9,en;q=0.8",
}


# ============================================================
# HELPERS
# ============================================================

def brand_slug(brand_entry: dict) -> str:
    """Extract the last path segment of the brand URL as a safe folder name."""
    return brand_entry["url"].rstrip("/").split("/")[-1]


def brand_folder(brand_entry: dict) -> Path:
    folder = ROOT_DIR / brand_slug(brand_entry)
    folder.mkdir(parents=True, exist_ok=True)
    return folder


def page_filename(brand_entry: dict, page_num: int) -> str:
    """
    Build a filename that parse_machines.py can reconstruct into a URL.
    Convention: <url_path_underscored>[_page_N].html
    Example: boutique-piece-detachee-climatisation_fournisseur_toshiba_page_2.html
    """
    path_part = brand_entry["url"].strip("/").replace("/", "_")
    if page_num == 1:
        return f"{path_part}.html"
    return f"{path_part}_page_{page_num}.html"


def extract_last_page(html: str) -> int:
    """Find the highest page number referenced in pagination links."""
    matches = re.findall(r'[?&]page=(\d+)', html)
    return max(map(int, matches), default=1)


def make_brand_url(brand_entry: dict, page_num: int) -> str:
    base = BASE_URL + brand_entry["url"]
    return base if page_num == 1 else f"{base}?page={page_num}"


# ============================================================
# FETCH
# ============================================================

async def fetch(session: aiohttp.ClientSession, url: str) -> str | None:
    for attempt in range(1, RETRIES + 1):
        try:
            async with session.get(url, headers=HEADERS, timeout=aiohttp.ClientTimeout(total=30)) as resp:
                if resp.status == 200:
                    return await resp.text()
                print(f"  ⚠️  HTTP {resp.status}  {url}")
        except asyncio.TimeoutError:
            print(f"  ⏱️  Timeout (attempt {attempt})  {url}")
        except Exception as exc:
            print(f"  ❌  Error (attempt {attempt})  {url}  →  {exc}")

        if attempt < RETRIES:
            await asyncio.sleep(RETRY_DELAY * attempt)

    return None


# ============================================================
# WORKER
# ============================================================

async def scrape_page(
    session: aiohttp.ClientSession,
    brand_entry: dict,
    page_num: int,
    semaphore: asyncio.Semaphore,
    folder: Path,
) -> bool:
    """Fetch and save one page. Returns True on success."""
    filepath = folder / page_filename(brand_entry, page_num)

    # Resume-safe: skip already downloaded pages
    if filepath.exists() and filepath.stat().st_size > 0:
        return True  # silently skip

    async with semaphore:
        url  = make_brand_url(brand_entry, page_num)
        html = await fetch(session, url)

    if html:
        filepath.write_text(html, encoding="utf-8")
        return True

    print(f"  ❌ Failed — {brand_entry['brand']} page {page_num}")
    return False


# ============================================================
# BRAND SCRAPER
# ============================================================

async def scrape_brand(session: aiohttp.ClientSession, brand_entry: dict) -> None:
    brand_name = brand_entry["brand"]
    folder     = brand_folder(brand_entry)
    semaphore  = asyncio.Semaphore(CONCURRENT_REQUESTS)

    print(f"\n{'='*60}")
    print(f"  🏷️  {brand_name}")
    print(f"{'='*60}")

    # Detect total pages from first page
    first_url  = make_brand_url(brand_entry, 1)
    first_html = await fetch(session, first_url)

    if not first_html:
        print(f"  ❌ Could not fetch first page for {brand_name} — skipping")
        return

    last_page = extract_last_page(first_html)
    print(f"  📄 Total pages: {last_page}")

    # Save page 1 immediately (already fetched)
    p1_path = folder / page_filename(brand_entry, 1)
    if not p1_path.exists() or p1_path.stat().st_size == 0:
        p1_path.write_text(first_html, encoding="utf-8")
        print(f"  ✅ Saved page 1")
    else:
        print(f"  ⏭️  Page 1 already exists")

    # Build tasks for pages 2..last_page
    tasks = [
        scrape_page(session, brand_entry, i, semaphore, folder)
        for i in range(2, last_page + 1)
    ]

    # Process in batches to avoid memory pressure on large brands
    ok = skipped = failed = 0
    ok += 1  # page 1

    for batch_start in range(0, len(tasks), BATCH_SIZE):
        batch = tasks[batch_start : batch_start + BATCH_SIZE]
        page_start = batch_start + 2
        page_end   = page_start + len(batch) - 1
        print(f"  🚀 Batch pages {page_start}–{page_end}  ({len(batch)} pages)")

        results = await asyncio.gather(*batch)
        ok      += sum(results)
        failed  += results.count(False)

        await asyncio.sleep(BATCH_DELAY)

    total = last_page
    print(f"\n  📊 {brand_name}: {ok}/{total} saved  |  {failed} failed")

    # Zip + download this brand
    zip_brand(brand_entry, brand_name)


# ============================================================
# ZIP + DOWNLOAD
# ============================================================

def zip_brand(brand_entry: dict, brand_name: str) -> Path:
    slug     = brand_slug(brand_entry)
    zip_path = ROOT_DIR / f"{slug}.zip"

    print(f"  📦 Zipping {brand_name}...")
    shutil.make_archive(str(zip_path.with_suffix("")), "zip", str(brand_folder(brand_entry)))
    print(f"  ✅ Zipped → {zip_path.name}  ({zip_path.stat().st_size / 1024:.1f} KB)")

    print(f"  ⬇️  Downloading {zip_path.name}...")
    # files.download(str(zip_path))
    return zip_path


def zip_all_brands() -> Path:
    zip_path = Path("all_brands.zip")
    print(f"\n{'='*60}")
    print(f"  📦 Zipping ALL brands into all_brands.zip ...")
    shutil.make_archive("all_brands", "zip", str(ROOT_DIR))
    print(f"  ✅ all_brands.zip  ({zip_path.stat().st_size / 1_048_576:.2f} MB)")
    print(f"  ⬇️  Downloading all_brands.zip ...")
    files.download("all_brands.zip")
    return zip_path


# ============================================================
# ENTRY POINT
# ============================================================

async def run_all(brands: list[dict] = BRANDS) -> None:
    """
    Scrape all brands sequentially (one brand at a time to be polite),
    zip+download each brand when done, then zip+download the full archive.

    You can pass a subset, e.g.:
        await run_all([BRANDS[0], BRANDS[1]])
    """
    connector = aiohttp.TCPConnector(
        limit=CONCURRENT_REQUESTS,
        ssl=False,            # avoid SSL issues common in Colab
        ttl_dns_cache=300,
    )

    t0 = time.time()

    async with aiohttp.ClientSession(connector=connector) as session:
        for i, brand_entry in enumerate(brands, 1):
            print(f"\n[{i}/{len(brands)}]", end="")
            await scrape_brand(session, brand_entry)

    elapsed = time.time() - t0
    print(f"\n⏱️  All brands done in {elapsed/60:.1f} min")

    # Final combined archive
    zip_all_brands()

# ============================================================
# HOW TO USE IN COLAB
# ============================================================
# Run the whole list:
#     await run_all()
#
# Run a single brand for testing:
#     await run_all([{"brand": "TOSHIBA", "url": "/boutique-piece-detachee-climatisation/fournisseur/toshiba"}])
#
# Run a slice:
#     await run_all(BRANDS[:5])

In [ ]:
await run_all()


[1/22]
  🏷️  CARRIER TRANSICOLD
  📄 Total pages: 1
  ✅ Saved page 1

  📊 CARRIER TRANSICOLD: 1/1 saved  |  0 failed
  📦 Zipping CARRIER TRANSICOLD...
  ✅ Zipped → carrier-transicold.zip  (6.1 KB)
  ⬇️  Downloading carrier-transicold.zip...

[2/22]
  🏷️  AIRCALO
  📄 Total pages: 1
  ✅ Saved page 1

  📊 AIRCALO: 1/1 saved  |  0 failed
  📦 Zipping AIRCALO...
  ✅ Zipped → aircalo.zip  (5.7 KB)
  ⬇️  Downloading aircalo.zip...

[3/22]
  🏷️  FRANCE AIR
  📄 Total pages: 1
  ✅ Saved page 1

  📊 FRANCE AIR: 1/1 saved  |  0 failed
  📦 Zipping FRANCE AIR...
  ✅ Zipped → france-air.zip  (5.7 KB)
  ⬇️  Downloading france-air.zip...

[4/22]
  🏷️  THERMOR
  📄 Total pages: 1
  ✅ Saved page 1

  📊 THERMOR: 1/1 saved  |  0 failed
  📦 Zipping THERMOR...
  ✅ Zipped → thermor.zip  (6.1 KB)
  ⬇️  Downloading thermor.zip...

[5/22]
  🏷️  SAMSUNG
  📄 Total pages: 1
  ✅ Saved page 1

  📊 SAMSUNG: 1/1 saved  |  0 failed
  📦 Zipping SAMSUNG...
  ✅ Zipped → samsung.zip  (6.3 KB)
  ⬇️  Downloading samsung.zip...


In [ ]:
import shutil
from google.colab import files

zip_file_name = 'general_pages.zip'
shutil.make_archive(zip_file_name.replace('.zip', ''), 'zip', 'general_pages')

print(f'Successfully created {zip_file_name}. Downloading now...')
files.download(zip_file_name)